# Naive Bayes on Play Tennis

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/play_tennis.csv")
print(df.shape)
df["play"].value_counts()

## Training: build the lookup table

In [ ]:
cols = ["outlook", "temperature", "humidity", "wind"]
class_counts = df["play"].value_counts()
prior = class_counts / len(df)
lookup = {col: pd.crosstab(df[col], df["play"]) / class_counts for col in cols}
print(prior.round(3).to_dict())
for col in cols:
    print(lookup[col].round(3), "\n")

## Testing: look up and multiply

In [ ]:
def predict(day, lookup=lookup, prior=prior):
    scores = {}
    for c in ["Yes", "No"]:
        s = prior[c]
        for col, value in day.items():
            s *= lookup[col].loc[value, c]
        scores[c] = s
    return scores

day1 = {"outlook": "Sunny", "temperature": "Hot", "humidity": "High", "wind": "Weak"}
s = predict(day1)
print({k: round(v, 5) for k, v in s.items()}, "-> play:", max(s, key=s.get))

## The zero-frequency problem

In [ ]:
day2 = {"outlook": "Overcast", "temperature": "Cool", "humidity": "Normal", "wind": "Weak"}
s = predict(day2)
print({k: round(v, 5) for k, v in s.items()})
print("P(Overcast | No) =", lookup["outlook"].loc["Overcast", "No"], "-> the No score is 0 whatever the other columns say")

## Laplace smoothing: add 1 to every count

In [ ]:
def smoothed_lookup(alpha=1):
    out = {}
    for col in cols:
        tab = pd.crosstab(df[col], df["play"]) + alpha
        out[col] = tab / tab.sum()                  # each class column sums to 1
    return out

lap = smoothed_lookup(1)
print(lap["outlook"].round(3))
for day in (day1, day2):
    s = predict(day, lookup=lap)
    total = sum(s.values())
    print({k: round(v / total, 3) for k, v in s.items()})

## scikit-learn: CategoricalNB

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()
X = enc.fit_transform(df[cols])
y = df["play"]
for alpha in (1e-10, 1.0):
    nb = CategoricalNB(alpha=alpha, min_categories=None).fit(X, y)
    for day in (day1, day2):
        q = enc.transform(pd.DataFrame([day]))
        print(f"alpha={alpha:g}", dict(zip(nb.classes_, nb.predict_proba(q)[0].round(3))))